# Llama 3.2 QLoRA fine-tuning (simple Colab version)
**Before running:** Runtime → Change runtime type → **T4 GPU**. Then run each cell top to bottom (Shift+Enter).

In [ ]:
# CELL 1 - install (no restart needed)
!pip install -q -U bitsandbytes peft accelerate
!pip install --upgrade torchao

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 107.3 MB/s eta 0:00:00
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
      Successfully uninstalled torchao-0.10.0


In [ ]:
# CELL 2 - imports + dataset
import json, os, random, torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model

assert torch.cuda.is_available(), "No GPU! Runtime > Change runtime type > T4 GPU, then run again."
random.seed(42); torch.manual_seed(42)

BASE_MODEL  = "unsloth/Llama-3.2-1B-Instruct"     # open copy of Llama 3.2 1B Instruct, no login needed
ADAPTER_DIR = "llama32_academic_lora"
SYSTEM = "You are an academic tutor. Answer with Definition, Explanation and Example."

def qa(q, d, e, x):
    return {"question": q, "answer": f"Definition: {d}\nExplanation: {e}\nExample: {x}"}

train_data = [
 qa("What is photosynthesis?", "Photosynthesis is the process by which plants, algae and some bacteria convert light energy into chemical energy stored in glucose.", "Chlorophyll absorbs sunlight; water and carbon dioxide are converted into glucose and oxygen.", "A leaf using sunlight to make sugar while releasing oxygen."),
 qa("What is Newton's second law of motion?", "The acceleration of an object is directly proportional to the net force on it and inversely proportional to its mass (F = ma).", "A larger force produces a larger acceleration, while a heavier object accelerates less for the same force.", "Pushing an empty shopping cart accelerates it more than pushing a full one."),
 qa("What is a prime number?", "A prime number is a natural number greater than 1 that has exactly two divisors: 1 and itself.", "Numbers such as 2, 3, 5 and 7 cannot be divided evenly by any other number.", "13 is prime because only 1 and 13 divide it evenly."),
 qa("What is the Pythagorean theorem?", "In a right triangle, the square of the hypotenuse equals the sum of the squares of the other two sides (a^2 + b^2 = c^2).", "It relates the three side lengths and is used to compute unknown distances.", "A triangle with sides 3 and 4 has hypotenuse 5."),
 qa("What is supply and demand?", "Supply and demand is an economic model describing how price is determined by the quantity sellers offer and the quantity buyers want.", "When demand rises and supply stays fixed, prices increase; when supply rises, prices tend to fall.", "Umbrella prices rise during a sudden rainy season."),
 qa("What is inflation?", "Inflation is the sustained increase in the general price level of goods and services over time.", "As prices rise, each unit of currency buys fewer goods, reducing purchasing power.", "If a loaf of bread rises from 20 to 22 rupees in a year, that reflects 10% inflation."),
 qa("What is DNA?", "DNA (deoxyribonucleic acid) is the molecule that carries genetic instructions for the growth and functioning of living organisms.", "It is a double helix built from four bases (A, T, C, G) whose sequence encodes genes.", "The DNA sequence of a gene determines the protein a cell produces."),
 qa("What is mitosis?", "Mitosis is a type of cell division that produces two genetically identical daughter cells.", "It has phases: prophase, metaphase, anaphase and telophase, followed by cytokinesis.", "Skin cells divide by mitosis to heal a cut."),
 qa("What is an algorithm?", "An algorithm is a finite, step-by-step procedure for solving a problem or performing a computation.", "It takes input, follows unambiguous steps and produces output, and is judged by correctness and efficiency.", "Binary search finds a name in a sorted list by repeatedly halving the search range."),
 qa("What is machine learning?", "Machine learning is a field of artificial intelligence in which systems learn patterns from data instead of being explicitly programmed.", "Models are trained on examples to minimize an error function and then make predictions on new data.", "A spam filter learns from labeled emails to classify new messages."),
 qa("What is a neural network?", "A neural network is a computational model made of layers of connected nodes (neurons) inspired by the brain.", "Each connection has a weight; training adjusts weights with backpropagation to reduce prediction error.", "An image classifier uses layers of neurons to recognize handwritten digits."),
 qa("What is the water cycle?", "The water cycle is the continuous movement of water between the oceans, atmosphere and land.", "Water evaporates, condenses into clouds, falls as precipitation and flows back through rivers and groundwater.", "Rain that falls on a hill eventually flows into a river and returns to the sea."),
 qa("What is gravity?", "Gravity is the force of attraction between objects with mass.", "Its strength grows with mass and weakens with the square of the distance between objects.", "Gravity keeps the Moon in orbit around the Earth."),
 qa("What is the French Revolution?", "The French Revolution (1789-1799) was a period of radical political and social change in France that ended the absolute monarchy.", "Economic hardship and Enlightenment ideas led to the fall of the Bastille and the rise of the republic.", "The storming of the Bastille in 1789 became a symbol of the revolution."),
 qa("What is democracy?", "Democracy is a system of government in which power is held by the people, directly or through elected representatives.", "Citizens vote in free elections, and rights and the rule of law limit government power.", "India holds general elections where citizens choose members of parliament."),
 qa("What is an atom?", "An atom is the smallest unit of a chemical element that keeps the element's properties.", "It has a nucleus of protons and neutrons surrounded by orbiting electrons.", "A carbon atom has 6 protons and 6 electrons."),
 qa("What is the periodic table?", "The periodic table is a tabular arrangement of chemical elements ordered by atomic number.", "Elements in the same column share similar chemical properties because of similar electron configurations.", "Sodium and potassium are in the same group and both react vigorously with water."),
 qa("What is an acid?", "An acid is a substance that donates hydrogen ions (H+) in water and has a pH below 7.", "Acids taste sour, turn blue litmus red and react with bases to form salts and water.", "Vinegar contains acetic acid."),
 qa("What is a derivative in calculus?", "A derivative measures the instantaneous rate of change of a function with respect to its variable.", "Geometrically it is the slope of the tangent line to the curve at a point.", "The derivative of x^2 is 2x."),
 qa("What is an integral in calculus?", "An integral computes the accumulated quantity under a curve, such as area.", "It is the reverse of differentiation, as stated by the Fundamental Theorem of Calculus.", "The integral of 2x is x^2 + C."),
 qa("What is probability?", "Probability is a number between 0 and 1 that measures how likely an event is to occur.", "It is computed as favorable outcomes divided by total equally likely outcomes.", "The probability of rolling a 4 on a fair die is 1/6."),
 qa("What is the mean of a data set?", "The mean is the sum of all values divided by the number of values.", "It measures the central tendency but is sensitive to extreme values.", "The mean of 2, 4 and 9 is 5."),
 qa("What is a database?", "A database is an organized collection of structured data stored and accessed electronically.", "A database management system lets users insert, query, update and secure the data.", "A university stores student records in a relational database."),
 qa("What is object-oriented programming?", "Object-oriented programming (OOP) is a paradigm that organizes software around objects that combine data and behavior.", "Its core principles are encapsulation, inheritance, polymorphism and abstraction.", "A Car class with speed and a drive() method can be inherited by ElectricCar."),
 qa("What is the greenhouse effect?", "The greenhouse effect is the warming of Earth as atmospheric gases trap heat radiated from the surface.", "Gases such as carbon dioxide and methane absorb infrared radiation and re-emit it downward.", "Rising CO2 from burning fossil fuels strengthens the effect and raises global temperatures."),
 qa("What is natural selection?", "Natural selection is the process by which organisms with advantageous traits survive and reproduce more than others.", "Over generations, favorable heritable traits become more common in a population.", "Darwin's finches developed different beak shapes suited to available food."),
 qa("What is a metaphor?", "A metaphor is a figure of speech that describes one thing as if it were another to suggest a similarity.", "Unlike a simile, it does not use 'like' or 'as'.", "\"Time is a thief\" implies that time steals moments from us."),
 qa("What is the Industrial Revolution?", "The Industrial Revolution was the transition from hand production to machine manufacturing beginning in Britain in the 18th century.", "Steam power, factories and new transport transformed economies and moved people to cities.", "The spinning jenny greatly increased textile output."),
 qa("What is GDP?", "Gross Domestic Product (GDP) is the total market value of all final goods and services produced in a country in a given period.", "Economists use it to measure the size and growth of an economy.", "If a country's GDP grows from 100 to 105 billion, it grew by 5%."),
 qa("What is osmosis?", "Osmosis is the movement of water across a semipermeable membrane from a region of low solute concentration to high solute concentration.", "It continues until concentrations equalize or pressure balances the flow.", "A raisin placed in water swells as water moves into it."),
 qa("What is Ohm's law?", "Ohm's law states that the current through a conductor is proportional to the voltage across it (V = IR).", "For a fixed resistance, doubling the voltage doubles the current.", "A 10 V battery across a 5 ohm resistor drives a 2 A current."),
 qa("What is recursion in computer science?", "Recursion is a technique in which a function solves a problem by calling itself on smaller subproblems.", "It needs a base case to stop and a recursive case that moves toward it.", "Factorial: n! = n * (n-1)!, with 0! = 1."),
 qa("What is cellular respiration?", "Cellular respiration is the process by which cells break down glucose to release usable energy as ATP.", "It includes glycolysis, the Krebs cycle and the electron transport chain, mostly in the mitochondria.", "Muscle cells use it to power a run."),
 qa("What is the Doppler effect?", "The Doppler effect is the change in observed frequency of a wave when the source and observer move relative to each other.", "Approaching sources compress waves (higher pitch); receding sources stretch them (lower pitch).", "An ambulance siren sounds higher as it approaches and lower as it leaves."),
]

# Unseen questions (NOT in the training set)
test_questions = [
 "What is entropy in thermodynamics?",
 "What is a linear regression?",
 "What is the Renaissance?",
 "What is meiosis?",
 "What is opportunity cost?",
 "What is a binary tree in computer science?",
]
train_titles = {d["question"] for d in train_data}
assert not (set(test_questions) & train_titles)


os.makedirs("data", exist_ok=True)
with open("data/academic_qa_train.jsonl", "w") as f:
    for d in train_data:
        f.write(json.dumps({"messages": [{"role": "system", "content": SYSTEM},
                                         {"role": "user", "content": d["question"]},
                                         {"role": "assistant", "content": d["answer"]}]}) + "\n")
print(len(train_data), "training examples saved -> data/academic_qa_train.jsonl")
print("Unseen test questions:", len(test_questions))

34 training examples saved -> data/academic_qa_train.jsonl
Unseen test questions: 6


In [ ]:
# CELL 3 - load base model in 4-bit and get ORIGINAL model answers
bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                         bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=torch.float32)
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

try:
    model = AutoModelForCausalLM.from_pretrained(BASE_MODEL, quantization_config=bnb, device_map={"": 0}, dtype=torch.float32)
except TypeError:
    model = AutoModelForCausalLM.from_pretrained(BASE_MODEL, quantization_config=bnb, device_map={"": 0}, torch_dtype=torch.float32)

EOT = tokenizer.convert_tokens_to_ids("<|eot_id|>")

def ask(question, max_new_tokens=200):
    msgs = [{"role": "system", "content": SYSTEM}, {"role": "user", "content": question}]
    text = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt", add_special_tokens=False).to("cuda")
    model.eval()
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False,
                             pad_token_id=tokenizer.pad_token_id, eos_token_id=[tokenizer.eos_token_id, EOT])
    return tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

base_answers = {q: ask(q) for q in test_questions}
for q, a in base_answers.items():
    print("Q:", q, "\nORIGINAL MODEL:", a, "\n" + "-"*70)

config.json:   0%|          | 0.00/894 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/54.7k [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.2MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/454 [00:00<?, ?B/s]

...

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

[transformers] Both `max_new_tokens` (=200) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.
[transformers] Both `max_new_tokens` (=200) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=200) a

Q: What is entropy in thermodynamics? 
ORIGINAL MODEL: **Definition:** Entropy in thermodynamics refers to a measure of disorder, randomness, or uncertainty in a system. It is a fundamental concept in thermodynamics that describes the degree to which a system's components are dispersed or mixed together.

**Explanation:** Entropy is a quantitative measure of the amount of thermal energy unavailable to do work in a system. It is a positive quantity, which means that as energy is transferred or transformed, entropy tends to increase. In other words, entropy increases when energy is wasted or lost as heat, sound, or other forms of energy.

**Example:** Consider a cup of hot coffee on a table. As the coffee cools, its molecules become more dispersed and lose their organized structure. This is an example of entropy increasing in a closed system (the coffee cup). The entropy of the coffee cup increases as the molecules become more random and dispersed.

**Key Concepts:**

* **Zeroth Law of T

In [ ]:
# CELL 4 - add LoRA and train (about 3-6 minutes)
model = get_peft_model(model, LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",
        target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"]))
model.print_trainable_parameters()

# tokenize: loss only on the answer part
examples = []
for d in train_data:
    msgs = [{"role": "system", "content": SYSTEM}, {"role": "user", "content": d["question"]}]
    prompt = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    p = tokenizer(prompt, add_special_tokens=False)["input_ids"]
    a = tokenizer(d["answer"] + "<|eot_id|>", add_special_tokens=False)["input_ids"]
    examples.append(((p + a)[:512], ([-100]*len(p) + a)[:512]))

def make_batch(items):
    m = max(len(x[0]) for x in items)
    ids = torch.full((len(items), m), tokenizer.pad_token_id); lab = torch.full((len(items), m), -100)
    att = torch.zeros((len(items), m), dtype=torch.long)
    for i, (x, y) in enumerate(items):
        ids[i, :len(x)] = torch.tensor(x); lab[i, :len(y)] = torch.tensor(y); att[i, :len(x)] = 1
    return {"input_ids": ids.cuda(), "labels": lab.cuda(), "attention_mask": att.cuda()}

params = [p for p in model.parameters() if p.requires_grad]
opt = torch.optim.AdamW(params, lr=2e-4)
EPOCHS, BS = 6, 4
model.train()
for epoch in range(EPOCHS):
    random.shuffle(examples); total = 0; n = 0
    for i in range(0, len(examples), BS):
        loss = model(**make_batch(examples[i:i+BS])).loss
        loss.backward()
        torch.nn.utils.clip_grad_norm_(params, 1.0)
        opt.step(); opt.zero_grad()
        total += loss.item(); n += 1
    print(f"Epoch {epoch+1}/{EPOCHS}  loss = {total/n:.4f}")
model.eval()

trainable params: 11,272,192 || all params: 1,247,086,592 || trainable%: 0.9039
Epoch 1/6  loss = 1.4243
Epoch 2/6  loss = 0.7307
Epoch 3/6  loss = 0.2890
Epoch 4/6  loss = 0.0587
Epoch 5/6  loss = 0.0315
Epoch 6/6  loss = 0.0258


PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): LlamaForCausalLM(
      (model): LlamaModel(
        (embed_tokens): Embedding(128256, 2048, padding_idx=128004)
        (layers): ModuleList(
          (0-15): 16 x LlamaDecoderLayer(
            (self_attn): LlamaAttention(
              (q_proj): lora.Linear4bit(
                (base_layer): Linear4bit(in_features=2048, out_features=2048, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.05, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=2048, out_features=16, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=16, out_features=2048, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
       

In [ ]:
# CELL 5 - save adapter, test on UNSEEN questions, compare
model.save_pretrained(ADAPTER_DIR); tokenizer.save_pretrained(ADAPTER_DIR)
print("Adapter saved in:", ADAPTER_DIR, os.listdir(ADAPTER_DIR), "\n")

tuned_answers = {q: ask(q) for q in test_questions}
ok = lambda t: all(k in t for k in ("Definition:", "Explanation:", "Example:"))
for q in test_questions:
    print("="*80, "\nQUESTION:", q)
    print("\n--- ORIGINAL ---\n" + base_answers[q])
    print("\n--- CUSTOMIZED (LoRA) ---\n" + tuned_answers[q])
print("\n" + "="*80)
print("Follows Definition/Explanation/Example format -> original: %d/%d | customized: %d/%d" % (
    sum(ok(base_answers[q]) for q in test_questions), len(test_questions),
    sum(ok(tuned_answers[q]) for q in test_questions), len(test_questions)))

[transformers] Both `max_new_tokens` (=200) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Adapter saved in: llama32_academic_lora ['adapter_config.json', 'README.md', 'adapter_model.safetensors', 'tokenizer_config.json', 'tokenizer.json', 'chat_template.jinja'] 



[transformers] Both `max_new_tokens` (=200) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=200) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=200) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=200) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https:/

QUESTION: What is entropy in thermodynamics?

--- ORIGINAL ---
**Definition:** Entropy in thermodynamics refers to a measure of disorder, randomness, or uncertainty in a system. It is a fundamental concept in thermodynamics that describes the degree to which a system's components are dispersed or mixed together.

**Explanation:** Entropy is a quantitative measure of the amount of thermal energy unavailable to do work in a system. It is a positive quantity, which means that as energy is transferred or transformed, entropy tends to increase. In other words, entropy increases when energy is wasted or lost as heat, sound, or other forms of energy.

**Example:** Consider a cup of hot coffee on a table. As the coffee cools, its molecules become more dispersed and lose their organized structure. This is an example of entropy increasing in a closed system (the coffee cup). The entropy of the coffee cup increases as the molecules become more random and dispersed.

**Key Concepts:**

* **Zeroth 

## Ollama: newer Ollama versions no longer load LoRA adapters, so the adapter is MERGED into the model and converted to GGUF.

In [ ]:
import gc
from peft import PeftModel
try:
    del model, opt
except NameError:
    pass # Variables 'model' or 'opt' were not defined, no need to delete them.
gc.collect(); torch.cuda.empty_cache()

try:
    base16 = AutoModelForCausalLM.from_pretrained(BASE_MODEL, dtype=torch.float16, device_map={"": "cpu"})
except TypeError:
    base16 = AutoModelForCausalLM.from_pretrained(BASE_MODEL, torch_dtype=torch.float16, device_map={"": "cpu"})
merged = PeftModel.from_pretrained(base16, ADAPTER_DIR).merge_and_unload()
merged.save_pretrained("llama32_academic_merged", safe_serialization=True)
tokenizer.save_pretrained("llama32_academic_merged")
del merged, base16; gc.collect()

!git clone -q --depth 1 https://github.com/ggml-org/llama.cpp
!pip install -q gguf sentencepiece protobuf
!python llama.cpp/convert_hf_to_gguf.py llama32_academic_merged --outfile academic-llama-f16.gguf --outtype f16
!ls -lh academic-llama-f16.gguf

modelfile = '''FROM ./academic-llama-f16.gguf
TEMPLATE """<|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 26 Jul 2024

{{ .System }}<|eot_id|><|start_header_id|>user<|end_header_id|>

{{ .Prompt }}<|eot_id|><|start_header_id|>assistant<|end_header_id|>

{{ .Response }}<|eot_id|>"""
SYSTEM """You are an academic tutor. Answer with Definition, Explanation and Example."""
PARAMETER stop "<|eot_id|>"
PARAMETER stop "<|start_header_id|>"
PARAMETER stop "<|end_header_id|>"
PARAMETER temperature 0.2
'''
open("Modelfile", "w").write(modelfile)
print("Modelfile written")

Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 118.5/118.5 kB 10.2 MB/s eta 0:00:00
INFO:hf-to-gguf:Loading model: llama32_academic_merged
INFO:numexpr.utils:NumExpr defaulting to 2 threads.
INFO:hf-to-gguf:Model architecture: LlamaForCausalLM
INFO:hf-to-gguf:gguf: indexing model part 'model.safetensors'
INFO:gguf.gguf_writer:gguf: This GGUF file is for Little Endian only
INFO:hf-to-gguf:Exporting model...
INFO:hf-to-gguf:rope_freqs.weight,           torch.float32 --> F32, shape = {32}
INFO:hf-to-gguf:token_embd.weight,           torch.float16 --> F16, shape = {2048, 128256}
INFO:hf-to-gguf:blk.0.attn_norm.weight,      torch.float16 --> F32, shape = {2048}
INFO:hf-to-gguf:blk.0.ffn_down.weight,       torch.float16 --> F16, shape = {8192, 2048}
INFO:hf-to-gguf:blk.0.ffn_gate.weight,       torch.float16 --> F16, shape = {2048, 8192}
INFO:hf-to-gguf:blk.0.ffn_up.weight,         torch.float16 --> F16, shape = {2048, 8192}
INFO:hf-to-gguf:blk.0.ffn_norm.weight,       torch.float16 --> F32, sha

In [ ]:
# CELL 7 - run Ollama inside Colab: original vs customized
!apt-get -qq install -y zstd pciutils > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh
import subprocess, time, requests
subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL); time.sleep(8)
!ollama pull llama3.2:1b
!ollama create academic-llama -f Modelfile
!ollama list

def ollama_ask(name, q):
    r = requests.post("http://localhost:11434/api/chat", json={"model": name, "stream": False, "options": {"temperature": 0},
        "messages": [{"role": "system", "content": SYSTEM}, {"role": "user", "content": q}]}, timeout=900)
    j = r.json()
    return j["message"]["content"] if "message" in j else f"[Ollama error: {j}]"

print("\n" + "="*80)
print("Testing with academic questions:\n")

academic_questions_to_test = [
    "What is a hash table in computer science?",
    "What is entropy in thermodynamics?",
    "What is quantum entanglement?",
    "Explain the concept of 'dark matter'.",
    "What is the significance of the Rosetta Stone?",
    "What is the Sapir-Whorf hypothesis?",
    "Describe the main principles of general relativity."
]

for q in academic_questions_to_test:
    print("=" * 80, "\nQUESTION:", q)
    print("\n--- ORIGINAL llama3.2:1b (Ollama) ---\n", ollama_ask("llama3.2:1b", q))
    print("\n--- CUSTOMIZED academic-llama (Ollama) ---\n", ollama_ask("academic-llama", q))
print("\n" + "="*80)

>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> NVIDIA GPU installed.
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


NAME                     ID              SIZE      MODIFIED               
academic-llama:latest    a2d38f9eae7f    2.5 GB    Less than a second ago    
llama3.2:1b              baf6a787fdff    1.3 GB    About a minute ago        

Testing with academic questions:

QUESTION: What is a hash table in computer science?

--- ORIGINAL llama3.2:1b (Ollama) ---
 **Definition:** A hash table (also known as a hash map or dictionary) is a data structure that stores key-value pairs in an array using a hash function to map keys to indices of the array.



In [13]:
# CELL 8 (optional) - download the small LoRA adapter to keep it
!zip -qr llama32_academic_lora.zip llama32_academic_lora
from google.colab import files
files.download("llama32_academic_lora.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>